In [ ]:
"""
Player Features Validation Test Notebook
=======================================

This notebook tests the player feature calculations to ensure:
1. Position filtering is working correctly
2. Rolling averages are calculated properly
3. Data alignment between teams and games is correct
4. Feature naming conventions are consistent
"""

import pandas as pd
import numpy as np
from src.models.xgboost_randomforrest_model_v2 import NFLModelV2
from src.config.config_v2 import player_feature_configs_v2

# Initialize model and build features
model = NFLModelV2(target="spread")
model.load_games(start_season=2024)  # Use recent data for testing
model.build_feature_matrices(include_defense=True, include_differentials=True)
player_features = model.build_player_feature_matrices(include_defense=True, include_differentials=True)

print("=== BASIC VALIDATION ===")
print(f"Total player features created: {len(player_features)}")
print(f"Historical DataFrame shape: {model._hist_df.shape}")
print()

In [ ]:
# Test 1: Position Filtering Validation
print("=== TEST 1: POSITION FILTERING ===")
qb1_features = [f for f in player_features if f.startswith('qb1_')]
rb1_features = [f for f in player_features if f.startswith('rb1_')]
wr1_features = [f for f in player_features if f.startswith('wr1_')]

print(f"QB1 features: {len(qb1_features)}")
print("Sample QB1 features:", qb1_features[:5])

print(f"RB1 features: {len(rb1_features)}")
print("Sample RB1 features:", rb1_features[:5])

print(f"WR1 features: {len(wr1_features)}")
print("Sample WR1 features:", wr1_features[:5])

# Check for unwanted features (noise)
qb1_receiving = [f for f in qb1_features if 'rec_' in f or 'targets' in f]
rb1_passing = [f for f in rb1_features if 'pass_' in f and 'pass_sacked' not in f]
wr1_passing = [f for f in wr1_features if 'pass_' in f]

print("\n--- NOISE DETECTION ---")
print(f"QB1 receiving features (should be empty): {len(qb1_receiving)}")
if qb1_receiving: print("WARNING:", qb1_receiving[:3])

print(f"RB1 passing features (should be empty): {len(rb1_passing)}")  
if rb1_passing: print("WARNING:", rb1_passing[:3])

print(f"WR1 passing features (should be empty): {len(wr1_passing)}")
if wr1_passing: print("WARNING:", wr1_passing[:3])
print()

In [ ]:
# %%
# 2. Explore raw player data before processing
print("Exploring raw player data...")

# Query to see Josh Allen's raw stats
josh_query = """
SELECT 
    g.season, g.week, g.date, o.player, o.teamid, 
    o.pass_yds, o.pass_att, o.pass_td, o.rush_yds, o.rush_att
FROM stats.offense o
JOIN stats.gamesummary g ON o.gamesummaryid = g.gamesummaryid
JOIN stats.player p ON o.playerid = p.playerid
WHERE o.player = 'Josh Allen' 
    AND g.season = 2024 
    AND p.current_position = 'QB'
ORDER BY g.week;
"""

josh_raw = execute_query(josh_query)
print("\nJosh Allen 2024 raw stats:")
print(josh_raw)

In [ ]:
# %%
# 3. Build team features first (prerequisite for player features)
print("Building team features...")
team_features = model.build_feature_matrices(include_defense=True, include_differentials=True)
print(f"Created {len(team_features)} team features")

In [ ]:
# %%
# 4. Add the player feature methods to the model (temporary for testing)
def add_player_methods_to_model(model):
    """Temporarily add player feature methods to model for testing."""
    
    def build_player_feature_matrices(self, exclude_current=True, include_defense=True, include_differentials=True):
        # [Your implementation from the artifact goes here]
        # For testing, let's do a simplified version first
        from src.config.config_v2 import player_feature_configs_v2
        
        # Get unique seasons from existing games
        seasons = self._hist_df['season'].unique()
        
        # Load player data for these seasons
        all_player_data = []
        for season in seasons:
            q = f"""
            SELECT 
                g.season, g.week, o.playerid, o.player, 
                p.current_position as position, o.teamid, o.gamesummaryid,
                g.hometeamid, g.awayteamid,
                COALESCE(o.pass_yds, 0) as pass_yds,
                COALESCE(o.pass_att, 0) as pass_att,
                COALESCE(o.rush_yds, 0) as rush_yds,
                COALESCE(o.rush_att, 0) as rush_att,
                COALESCE(o.rec_yds, 0) as rec_yds,
                COALESCE(o.targets, 0) as targets
            FROM stats.offense o
            JOIN stats.gamesummary g ON o.gamesummaryid = g.gamesummaryid
            JOIN stats.player p ON o.playerid = p.playerid
            WHERE g.season = {season}
            ORDER BY g.season, g.week;
            """
            
            player_data = execute_query(q)
            if player_data is not None and not player_data.empty:
                all_player_data.append(player_data)
        
        if not all_player_data:
            return []
            
        player_df = pd.concat(all_player_data, ignore_index=True)
        
        # Add defensive opponent team id
        player_df["def_teamid"] = np.where(
            player_df["teamid"] == player_df["hometeamid"], 
            player_df["awayteamid"], 
            player_df["hometeamid"]
        )
        
        # Get top players by position for each team/game
        top_players = self._get_top_players_by_game(player_df)
        
        # For testing, let's just focus on QB1 pass_yds
        qb1_data = top_players[top_players['player_rank'] == 'QB1'].copy()
        
        if qb1_data.empty:
            return []
        
        # Sort and create rolling averages
        qb1_data = qb1_data.sort_values(['teamid', 'season', 'week']).reset_index(drop=True)
        
        # Manual rolling calculation for verification
        def prior_rolling(series, w):
            s = series.shift(1 if exclude_current else 0)
            return s.rolling(window=w, min_periods=1).mean()
        
        # Group by team and calculate rolling averages
        team_grp = qb1_data.groupby('teamid', group_keys=False)
        qb1_data['qb1_pass_yds_roll5'] = team_grp['pass_yds'].apply(lambda s: prior_rolling(s, 5))
        
        # Store for inspection
        self._test_qb1_data = qb1_data
        
        return ['qb1_pass_yds_roll5']
    
    def _get_top_players_by_game(self, df):
        """Identify top players by position for each team/game."""
        position_rankings = []
        
        # Group by game and team
        for (season, week, gamesummaryid, teamid), team_game in df.groupby(['season', 'week', 'gamesummaryid', 'teamid']):
            
            # QB1 - top QB by pass attempts
            qbs = team_game[team_game['position'] == 'QB']
            if not qbs.empty:
                qb1 = qbs.nlargest(1, 'pass_att').copy()
                qb1['player_rank'] = 'QB1'
                position_rankings.append(qb1)
        
        if position_rankings:
            return pd.concat(position_rankings, ignore_index=True)
        else:
            return pd.DataFrame()
    
    # Add methods to model
    import types
    model.build_player_feature_matrices = types.MethodType(build_player_feature_matrices, model)
    model._get_top_players_by_game = types.MethodType(_get_top_players_by_game, model)

# Add the methods
add_player_methods_to_model(model)

In [ ]:
# 5. Build player features and inspect results
print("Building player features...")
player_features = model.build_player_feature_matrices(exclude_current=True, include_defense=False)
print(f"Created {len(player_features)} player features")


In [ ]:

# %%
# 6. Examine QB1 data for validation
if hasattr(model, '_test_qb1_data'):
    qb1_test = model._test_qb1_data
    
    # Focus on Josh Allen for manual verification
    josh_qb1 = qb1_test[qb1_test['player'] == 'Josh Allen'].copy()
    
    print("\nJosh Allen QB1 rolling average test:")
    print("Week | Pass Yds | 5-Game Roll | Manual Check")
    print("-" * 50)
    
    for i in range(min(8, len(josh_qb1))):
        row = josh_qb1.iloc[i]
        week = row['week']
        pass_yds = row['pass_yds']
        roll_5 = row['qb1_pass_yds_roll5']
        
        # Manual calculation for verification
        if i == 0:
            manual = np.nan
        else:
            prior_games = josh_qb1.iloc[max(0, i-5):i]
            manual = prior_games['pass_yds'].mean()
        
        match = "✓" if abs(roll_5 - manual) < 0.01 or (pd.isna(roll_5) and pd.isna(manual)) else "✗"
        print(f"{week:4d} | {pass_yds:8.0f} | {roll_5:10.1f} | {manual:7.1f} {match}")

In [ ]:
# 7. Visualize rolling average to spot-check visually
if hasattr(model, '_test_qb1_data'):
    josh_qb1 = model._test_qb1_data[model._test_qb1_data['player'] == 'Josh Allen'].copy()
    
    plt.figure(figsize=(12, 6))
    plt.plot(josh_qb1['week'], josh_qb1['pass_yds'], 'o-', label='Actual Pass Yds', linewidth=2)
    plt.plot(josh_qb1['week'], josh_qb1['qb1_pass_yds_roll5'], 's-', label='5-Game Rolling Avg', linewidth=2)
    plt.xlabel('Week')
    plt.ylabel('Passing Yards')
    plt.title('Josh Allen 2024: Actual vs 5-Game Rolling Average')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
# %%
# 8. Test different teams to verify ranking logic
print("\n=== Testing Player Rankings ===")

# Check Week 5 for multiple teams
week5_query = """
SELECT 
    g.season, g.week, o.teamid, o.player, 
    p.current_position, o.pass_att, o.rush_att, o.targets
FROM stats.offense o
JOIN stats.gamesummary g ON o.gamesummaryid = g.gamesummaryid
JOIN stats.player p ON o.playerid = p.playerid
WHERE g.season = 2024 AND g.week = 5 
    AND o.teamid IN ('BUF', 'KC', 'SF')
ORDER BY o.teamid, p.current_position, o.pass_att DESC, o.rush_att DESC, o.targets DESC;
"""

week5_raw = execute_query(week5_query)
print("\nWeek 5 raw stats for BUF, KC, SF:")
print(week5_raw[['teamid', 'player', 'current_position', 'pass_att', 'rush_att', 'targets']])

# Compare with our QB1 selections
if hasattr(model, '_test_qb1_data'):
    week5_qb1 = model._test_qb1_data[model._test_qb1_data['week'] == 5]
    print("\nOur QB1 selections for Week 5:")
    print(week5_qb1[['teamid', 'player', 'pass_att']])

In [ ]:
# %%
# 9. Test exclude_current logic by comparing with include_current
print("\n=== Testing Exclude Current Logic ===")

# Build model with exclude_current=False for comparison
model_include = NFLModelV2(target="spread")
model_include.load_games(start_season=2024)
model_include.build_feature_matrices(include_defense=True, include_differentials=True)
add_player_methods_to_model(model_include)
model_include.build_player_feature_matrices(exclude_current=False, include_defense=False)

# Compare the first few games
if hasattr(model, '_test_qb1_data') and hasattr(model_include, '_test_qb1_data'):
    josh_exclude = model._test_qb1_data[model._test_qb1_data['player'] == 'Josh Allen'].copy()
    josh_include = model_include._test_qb1_data[model_include._test_qb1_data['player'] == 'Josh Allen'].copy()
    
    print("Josh Allen rolling averages: Exclude vs Include Current Game")
    print("Week | Exclude Current | Include Current | Difference")
    print("-" * 55)
    
    for i in range(min(6, len(josh_exclude))):
        week = josh_exclude.iloc[i]['week']
        exclude_val = josh_exclude.iloc[i]['qb1_pass_yds_roll5']
        include_val = josh_include.iloc[i]['qb1_pass_yds_roll5']
        diff = include_val - exclude_val
        
        print(f"{week:4d} | {exclude_val:14.1f} | {include_val:14.1f} | {diff:9.1f}")

print("\n=== Test Complete ===")
print("Use the visualizations and manual calculations above to verify the rolling averages are correct.")